# Etapa 2, E4 krok B (osoba 2): prompt s príkladmi z trénovacích dát

Analýza chýb prvého behu (prompt v1) ukázala, že LLM nepozná konvencie označovania datasetu: namiesto konkrétneho alergénu („milk and products thereof“, 588 správ v train) volí všeobecné triedy („undeclared constituent“ 7 správ, „allergens“ 13 správ). Preto dve nové verzie promptu:

- **v2:** ku každej možnosti počet trénovacích správ s touto triedou a jeden ich titulok,
- **v3:** v2 + informácia, že možnosti zoradil klasifikátor a A je jeho najlepší tip.

Model aj nainštalované balíčky z prvého behu sa použijú znova. Všetko je na **valid**.

| Bunka | Čo robí | Čas |
| --- | --- | --- |
| 1 | nastavenie, kontrola nahraných súborov | hneď |
| 2 | inštalácia torch a transformers (len ak po reštarte chýbajú) | 0 alebo 3 až 5 min |
| 3 | kontrola GPU a modelu | pár sekúnd |
| 4 | prompt v2 na celom valid | asi 8 min |
| 5 | prompt v3 na celom valid | asi 8 min |
| 6 | zbalenie výsledkov do `vysledky_e4b.zip` | hneď |

Spúšťaj bunky postupne zhora (Shift+Enter). Odpovede sa ukladajú priebežne; po prerušení spusti bunky 1, 3 a tú, ktorá bežala.

**Bunka 1:** rovnaké nastavenie ako v prvom notebooku. Všetky súbory musia byť `OK` (nový je `data/incidents_train.csv`, z neho sa berú počty a príklady).

In [ ]:
import glob, importlib.util, os, sys, time, zipfile

# >>> nastavenie (rovnaké ako v etapa2_e4_llm_datalab.ipynb) <<<
PRIECINOK_MODELOV = "/home/jovyan/work/SpracovaniePrirodzenehoJazyka/zadanie_2etapa/llm_modely"
MODEL = "Qwen/Qwen2.5-3B-Instruct"

MODEL_CESTA = os.path.join(PRIECINOK_MODELOV, MODEL.split("/")[-1])
os.environ["HF_HOME"] = os.path.join(PRIECINOK_MODELOV, "hf_home")
os.environ["PIP_NO_CACHE_DIR"] = "1"
KOREN = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(KOREN)
print("koreň projektu:", KOREN)
potrebne = ["src/spolocne.py", "src/e3_hierarchia.py", "src/e4_llm.py", "data/incidents_train.csv",
            "data/incidents_valid.csv", "results/etapa2/kandidati/E12kens_valid.csv",
            "results/etapa2/predikcie/E12kens_valid.csv", "requirements_e4.txt"]
for s in potrebne:
    print("OK    " if os.path.exists(s) else "CHÝBA ", s)
assert all(os.path.exists(s) for s in potrebne), "niektorý súbor chýba, skontroluj štruktúru priečinkov"

**Bunka 2:** ak torch a transformers po reštarte servera zmizli, nainštaluje ich znova (varovanie o `httpcore`/`h11` je neškodné, ako pri prvom behu).

In [ ]:
chybaju = [b for b in ("torch", "transformers") if importlib.util.find_spec(b) is None]
if chybaju:
    print("inštalujem:", chybaju)
    !{sys.executable} -m pip install --no-cache-dir -q -r requirements_e4.txt
else:
    print("torch a transformers už sú nainštalované")

**Bunka 3:** GPU a model. Musí vypísať názov karty a `model nájdený`. Ak model chýba (napr. bol zmazaný), stiahne sa znova do `PRIECINOK_MODELOV`.

In [ ]:
import torch, transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "torch nevidí GPU: je zvolená konfigurácia s GPU?"
print("GPU:", torch.cuda.get_device_name(0))
if not glob.glob(os.path.join(MODEL_CESTA, "*.safetensors")):
    from huggingface_hub import snapshot_download
    print("model chýba, sťahujem …")
    snapshot_download(MODEL, local_dir=MODEL_CESTA,
                      allow_patterns=["*.json", "*.safetensors", "*.txt", "tokenizer*", "LICENSE*"])
print("model nájdený:", MODEL_CESTA)

**Bunka 4:** prompt v2 na celom valid (565 správ × 2 štítky). Na začiatku sa vypíše ukážka promptu pre prvú správu. Prompt je asi o 300 tokenov dlhší, preto beh trvá dlhšie ako v1.

In [ ]:
!{sys.executable} src/e4_llm.py --backend hf --model {MODEL} --model-cesta {MODEL_CESTA} --prompt v2 --ukaz-prompt

**Bunka 5:** prompt v3 (v2 + informácia o poradí kandidátov) na celom valid.

In [ ]:
!{sys.executable} src/e4_llm.py --backend hf --model {MODEL} --model-cesta {MODEL_CESTA} --prompt v3

**Bunka 6:** zbalí výsledky do `vysledky_e4b.zip` v koreni projektu (`NLP_food_hazard/`). Stiahni ho a vlož do koreňa projektu na svojom PC; vyhodnotenie (krížová validácia váhy, bootstrap) urobíme tam, GPU na to netreba.

In [ ]:
subory = sorted(glob.glob("results/etapa2/llm_cache/*") + glob.glob("results/etapa2/llm/detail_*_v[23]_*")
                + glob.glob("results/etapa2/predikcie/E4_*_v[23]_*") + glob.glob("results/etapa2/logy/E4_*_v[23]_*"))
with zipfile.ZipFile("vysledky_e4b.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for s in subory:
        z.write(s)
        print(s)
print(f"
{len(subory)} súborov v {os.path.join(KOREN, 'vysledky_e4b.zip')}")